---
title: Methodology & data
description: "How the DECE Research Analytics site was built: Scopus and ROR data, the processing pipeline, affiliation matching and its validation, definitions, data quality and limitations."
---

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "dece_stats").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from dece_stats import config, data, periods
from dece_stats import plotting as dp

dp.apply_theme()
PARTIAL = config.PARTIAL_YEAR   # the year of the export, covered only up to the export date
AS_OF = config.EXPORT_DATE      # the export date
LAST_FULL = periods.LAST_FULL   # the latest full year, PARTIAL - 1

In [ ]:
import json
import math
import re
from statistics import NormalDist

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dece_stats import scopus, small_groups as sg

# ===================================================================== inputs
meta = data.load_metadata()
pubs = data.load_publications()
aff = data.load_affiliations()
pub_aff = data.load_publication_affiliations()
authors = data.load_publication_authors()
countries = data.load_countries()
orgs = data.load_organizations()      # ROR types of the partner organisations (industry partners)
# The matching validation lives in data/validation (dece_stats.data has no loader for it):
# summary.json holds the verdict counts, matching_validation.csv one row per judged string.
validation = json.loads((config.VALIDATION_DIR / "summary.json").read_text(encoding="utf-8"))
judged = pd.read_csv(config.VALIDATION_DIR / "matching_validation.csv", dtype=str,
                     keep_default_na=False, na_values=[""], encoding="utf-8")


def fmt_date(value) -> str:
    """ISO date text (or a date) -> day without a leading zero, month name and year."""
    ts = pd.Timestamp(value)
    return f"{ts.day} {ts:%B %Y}"


def ci(bounds) -> str:
    """Interval of two ratios as text: (0.9741, 0.9992) -> '97.4–99.9%'."""
    lo, hi = bounds
    return f"{dp.fmt_pct(lo)[:-1]}–{dp.fmt_pct(hi)}"


def fmt_share(value: float) -> str:
    """dp.fmt_pct that never shows '100.0%' for a share below 1 (nor '0.0%' above 0)."""
    text = dp.fmt_pct(value)
    if text == "100.0%" and value < 1:
        return ">99.9%"
    if text == "0.0%" and value > 0:
        return "<0.1%"
    return text


def n_of(n: int, noun: str) -> str:
    """'1 string', '13 strings'."""
    return f"{dp.fmt_int(n)} {noun if n == 1 else noun + 's'}"


def join_and(items) -> str:
    """['a', 'b', 'c'] -> 'a, b and c'."""
    items = [str(i) for i in items]
    return items[0] if len(items) == 1 else ", ".join(items[:-1]) + " and " + items[-1]


# ===================================================================== sources
n_pubs = len(pubs)
year_min, year_max = int(pubs["year"].min()), int(pubs["year"].max())
n_partial = int(pubs["is_partial_year"].sum())
as_of_text = fmt_date(AS_OF)
as_of_short = f"{AS_OF.day} {AS_OF:%B}"
query = meta["scopus_query"]
ror_version = meta["ror_version"]
ror_date_text = fmt_date(meta["ror_date"])
generated_text = fmt_date(meta["generated_at"][:10])
n_countries = len(countries)
ne_scale = re.search(r"_(\d+m)_", config.NATURAL_EARTH.name).group(1)   # "110m"
assert meta["export_date"] == AS_OF.isoformat(), "text: the export date (data/processed is from another export)"
assert query == config.SCOPUS_QUERY and config.SCOPUS_QUERY_WORDS == (
    ("elec", "elect", "electr", "electrical"), ("aristotle", "aristotelian")), "text: the query words (Data sources)"
assert {"MT", "SG"} <= set(countries["country_code"]), "text: Malta and Singapore, countries in the data"
assert not [c for c in pubs.columns if re.search("abstract|keyword", c)], \
    "text: abstracts and keywords are not kept, so topics come from titles alone"

# Record selection (dece_stats.pipeline.select_records). Scopus may apply the query across the
# affiliations of a record. The pipeline keeps the records in which one affiliation holds a word of every
# group of the query, or no affiliation shown holds any of the words, unless they are dated after the
# export year; metadata.json counts each kind ("records").
rec = meta["records"]
n_exported, n_split = rec["exported"], rec["dropped_query_split"]
assert n_exported == (rec["query_in_one_affiliation"] + rec["query_in_profile_only"] + n_split
                      + rec["dropped_no_affiliation"]), "metadata.json: record counts"
assert rec["kept"] == n_pubs == (n_exported - n_split - rec["dropped_no_affiliation"]
                                 - rec["dropped_after_export_year"]), "metadata.json: record counts"
assert n_split > 0, "text: Scopus applies the query across the affiliations of a record (records left out)"
share_split = n_split / n_exported
LEFT_OUT = [  # (records, reason for one, reason for several)
    (n_split, "because its query words occur only in different affiliations",
     "because the query words occur only in different affiliations"),
    (rec["dropped_no_affiliation"], "because it has no affiliations", "because they have no affiliations"),
    (rec["dropped_after_export_year"], f"because it is dated after {PARTIAL}",
     f"because they are dated after {PARTIAL}"),
]
left_out_text = join_and([
    f"{dp.fmt_int(n)} {'was' if n == 1 else 'were'} left out {one if n == 1 else many}" if i == 0
    else f"{dp.fmt_int(n)} {one if n == 1 else many}"
    for i, (n, one, many) in enumerate(row for row in LEFT_OUT if row[0])])
# The same rule applied to the affiliations of the records kept (in their order in the record)
aff_field = pub_aff.sort_values(["eid", "aff_pos"]).groupby("eid")["affiliation"].agg("; ".join)
scope = pubs["eid"].map(aff_field).map(lambda value: scopus.query_scope(value, config.SCOPUS_QUERY_WORDS))
n_profile = int(scope.eq("profile").sum())       # no affiliation shown holds any of the words
assert scope.isin(["one", "profile"]).all() and n_profile <= rec["query_in_profile_only"], \
    "text: the records kept have the query words in one affiliation, or none of them"
kept_text = "those with the query words in one affiliation" + (
    f" and {n_of(n_profile, 'record')} whose affiliations show none of the words, which Scopus matched "
    "through affiliation data that the export does not show" if n_profile else "")

# ===================================================================== periods
# Five-year periods that end with the latest full year (dece_stats.periods): P_RECENT holds the latest
# five full years, and a newer export moves every period along.
P_RECENT = periods.period_of(LAST_FULL)          # "recent"
P_PREV = periods.period_of(LAST_FULL - 5)        # the period before P_RECENT
P_EARLIER = periods.period_of(LAST_FULL - 10)    # the period ten years before P_RECENT
RECENT, EARLIER = periods.bounds(P_RECENT), periods.bounds(P_EARLIER)
first_period = periods.period_of(year_min, first_year=year_min)
first_text = first_period + ("" if periods.bounds(first_period)[1] - year_min + 1 == periods.WIDTH
                             else f", which is shorter because the data start in {year_min}")
# The period charts on this page (fixed choices): the periods that start in MODERN_FROM or later, the
# period before them, the sparse years before that merged into one column, and the partial year.
MODERN_FROM = 2000
full_periods = periods.periods(year_min)
modern = [p for p in full_periods if periods.bounds(p)[0] >= MODERN_FROM]
assert full_periods.index(modern[0]) >= 2 and modern[-1] == P_RECENT, "period charts: no early years to merge"
second_period = full_periods[full_periods.index(modern[0]) - 1]
EARLY_END = periods.bounds(second_period)[0] - 1
early_label = f"{year_min}–{EARLY_END}"
PERIODS = [early_label, second_period, *modern, periods.period_of(PARTIAL)]


def chart_period(year: int) -> str:
    """Column of the period charts: the shared period of the year, the years to EARLY_END merged."""
    return early_label if year <= EARLY_END else periods.period_of(year)


# ===================================================================== pipeline tables
TABLES = {
    "publications": "Scopus record (key: EID), with the derived paper-level fields",
    "affiliations": "Distinct affiliation string, with its country and ROR match",
    "publication_affiliations": "Affiliation string in a record, with its match and holders",
    "organizations": "ROR organisation used by a match, or an ancestor of one",
    "countries": "Country in the data: ISO codes, continent, UN region, EU-27 flag",
    "publication_authors": "Author position in a record: Scopus author id and DECE flag (no names)",
    "publication_funders": "Funding entry of a record: funder, acronym, grant numbers",
}
tables = pd.DataFrame({
    "file": [config.PROCESSED_FILES[k] for k in TABLES],
    "rows": [int(meta["rows"][k]) for k in TABLES],
    "row": list(TABLES.values()),
})

# ===================================================================== affiliation matching
mm = meta["matching"]
MATCHED = ("override", "exact", "acronym", "fuzzy")
n_strings = int(meta["rows"]["affiliations"])
n_occ = int(meta["rows"]["publication_affiliations"])
n_ignored = int(mm["methods_unique"]["ignored"])
n_override = int(mm["methods_unique"]["override"])
strings_usable, strings_matched = mm["all"]["unique_strings"], mm["all"]["unique_matched"]
occ_usable, occ_matched = mm["all"]["occurrences"], mm["all"]["occurrences_matched"]
str_share, occ_share = strings_matched / strings_usable, occ_matched / occ_usable
assert (strings_usable == n_strings - n_ignored
        and occ_usable == n_occ - int(mm["methods_occurrences"]["ignored"])), \
    "text: the matching shares count every affiliation except the society memberships (KPI note, key findings)"
ext = mm["excluding_auth"]
str_share_ext = ext["unique_matched"] / ext["unique_strings"]
occ_share_ext = ext["occurrences_matched"] / ext["occurrences"]
cut, cut_neutral, cut_nocountry = (dp.fmt_num(mm[k], 0) for k in
                                   ("fuzzy_cutoff", "fuzzy_cutoff_neutral", "fuzzy_cutoff_no_country"))

METHOD_ROWS = {
    "exact": "Exact name, alias or label",
    "acronym": "Acronym, unique in the country",
    "fuzzy": "Fuzzy name match",
    "override": "Manual override",
    "none": "Not matched",
    "ignored": "Ignored: society membership only",
}
methods = pd.DataFrame({
    "outcome": list(METHOD_ROWS.values()),
    "strings": [int(mm["methods_unique"][k]) for k in METHOD_ROWS],
    "occurrences": [int(mm["methods_occurrences"][k]) for k in METHOD_ROWS],
    # share of the occurrences that are not ignored (the coverage denominator)
    "share": [mm["methods_occurrences"][k] / occ_usable if k != "ignored" else np.nan
              for k in METHOD_ROWS],
})

usable = pub_aff[pub_aff["match_method"] != "ignored"].assign(
    matched=lambda d: d["match_method"].isin(MATCHED),
    period=lambda d: d["year"].map(chart_period),
)
partner_occ = usable[~usable["is_auth"]]


def coverage(df: pd.DataFrame) -> pd.DataFrame:
    """Matched share of affiliation occurrences per period: n, k (matched), share."""
    out = df.groupby("period")["matched"].agg(n="size", k="sum").reindex(PERIODS)
    out["share"] = out["k"] / out["n"]
    return out


cov_all = coverage(usable)
cov_ext = coverage(partner_occ)
ext_min, ext_max = cov_ext.loc[modern, "share"].min(), cov_ext.loc[modern, "share"].max()
ext_second = cov_ext.loc[second_period, "share"]
ext_early, n_ext_early = cov_ext.loc[early_label, "share"], int(cov_ext.loc[early_label, "n"])
ext_pre_modern = partner_occ.loc[partner_occ["year"] < MODERN_FROM, "matched"].mean()
assert max(ext_second, ext_early) < ext_min, "text: partner counts for the older periods are less complete"
assert ext_pre_modern < occ_share_ext, "text: outside AUTH the share drops in the records before MODERN_FROM"
assert occ_share > str_share, "text: counted by occurrence the share is higher"
assert str_share_ext < str_share and occ_share_ext < occ_share, "text: coverage is lower outside AUTH"
partner_tops = partner_occ.loc[partner_occ["ror_id_top"].notna()
                               & partner_occ["ror_id_top"].ne(config.AUTH_ROR), "ror_id_top"]
n_partner_orgs = int(partner_tops.nunique())

# ===================================================================== matching validation
# Rates and 95% Wilson intervals are recomputed from the verdict counts in summary.json
# (its stored rates are rounded to four decimals; recomputing avoids double rounding).
Z95 = NormalDist().inv_cdf(0.975)


def wilson(k: int, n: int, z: float = Z95) -> tuple[float, float]:
    """Wilson score interval for k successes in n trials."""
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return centre - half, centre + half


def rates(counts: dict, weighted: dict | None = None, judged_n: int | None = None) -> dict:
    """Precision = correct / (correct + wrong); recall = correct / (correct + wrong + missed);
    Wilson intervals on the per-string counts; paper-weighted variants from `weighted`."""
    k, w, m = (int(counts.get(v, 0)) for v in ("correct", "wrong", "missed"))
    out = {**{v: int(counts.get(v, 0)) for v in
              ("correct", "wrong", "missed", "correct_unmatched", "uncertain")},
           "judged": judged_n, "n_p": k + w, "n_r": k + w + m,
           "precision": k / (k + w), "precision_ci": wilson(k, k + w),
           "recall": k / (k + w + m), "recall_ci": wilson(k, k + w + m)}
    if weighted is not None:
        kw, ww, mw = (weighted.get(v, 0) for v in ("correct", "wrong", "missed"))
        out.update(precision_w=kw / (kw + ww), recall_w=kw / (kw + ww + mw), missed_papers=int(mw))
    return out


def summary_rates(entry: dict) -> dict:
    return rates(entry["counts"], entry["weighted_counts"], entry["judged"])


# The round-2 verdicts were made with ROR v1.66. After a ROR update the strings whose match changed are
# judged again and the missed and no-record verdicts re-checked (data/validation/round2_recheck.csv):
# the final matcher's round-2 figures are those of the current dump, new2_judged those first judged.
recheck = validation["round2_recheck"]
assert recheck["ror_version"] == meta["ror_version"], (
    f"the validation re-check is for ROR {recheck['ror_version']}, the data use {meta['ror_version']}: "
    "judge the round-2 strings whose match changed again (data/validation/README.md)")
recheck_rows = pd.read_csv(config.VALIDATION_DIR / "round2_recheck.csv", dtype=str, keep_default_na=False,
                           na_values=[""], encoding="utf-8").set_index("row_id")
rc_ror, rc_date, rc_n = recheck["ror_version"], fmt_date(recheck["ror_date"]), recheck["rechecked"]
is_changed = recheck_rows["match_changed"].eq("True")
rc_changed, rc_others = recheck_rows[is_changed], recheck_rows[~is_changed]
assert len(rc_changed) == recheck["changed"] and len(recheck_rows) == rc_n, \
    "round2_recheck.csv and summary.json differ"
assert len(rc_changed) > 0, "text: the ROR update changed the final matcher's result for some round-2 strings"
# How the result changed (top_name_before, top_name_now: the top-level organisation returned with v1.66 and now)
was_matched, is_matched = rc_changed["top_name_before"].notna(), rc_changed["top_name_now"].notna()
assert (was_matched | is_matched).all(), "round2_recheck.csv: a changed result without a match before or now"
rc_moves_text = join_and([f"{verb} {dp.fmt_int(n)}{tail}" for n, verb, tail in (
    (int((~was_matched & is_matched).sum()), "matches", " that it had left unmatched"),
    (int((was_matched & is_matched).sum()), "returns another organisation for", ""),
    (int((was_matched & ~is_matched).sum()), "no longer matches", "")) if n])
# The strings re-checked: those whose result changed and all those judged missed or without a ROR record
v166 = judged[judged["round"] == "2"].set_index("row_id")["new_verdict"]
missed_or_noror = v166.index[v166.isin(["missed", "correct_unmatched"])]
assert set(recheck_rows.index) == set(rc_changed.index) | set(missed_or_noror), \
    "text: the strings re-checked are the changed ones and those judged missed or without a ROR record"
rc_now_correct = int(rc_changed["verdict_now"].eq("correct").sum())
rc_before = rc_changed.loc[rc_changed["verdict_now"].eq("correct"), "verdict_before"].value_counts()
assert rc_changed.loc[rc_changed["verdict_before"].eq("correct") & rc_changed["verdict_now"].eq("correct"),
                      "note"].str.contains("successor", case=False).all(), \
    "text: the strings correct before and now go to the organisation's ROR successor"
rc_parts = [f"{dp.fmt_int(n)} {one if n == 1 else many}" for n, one, many in (
    (rc_before.get("missed", 0), "had been missed", "had been missed"),
    (rc_before.get("correct_unmatched", 0), "names an organisation that ROR v1.66 did not list",
     "name organisations that ROR v1.66 did not list"),
    (rc_before.get("correct", 0), "was correct before and now goes to the organisation's ROR successor",
     "were correct before and now go to the organisations' ROR successors")) if n]
# the other re-checked strings (result unchanged): how many keep their verdict
rc_stand = int((rc_others["verdict_now"] == rc_others["verdict_before"]).sum())
rc_text = (f"{'all ' if rc_now_correct == len(rc_changed) else ''}{dp.fmt_int(rc_now_correct)} of the changed "
           f"strings are now correct ({join_and(rc_parts)})"
           + (f", and the verdicts of {'the' if rc_stand == len(rc_others) else dp.fmt_int(rc_stand) + ' of the'} "
              f"other {n_of(len(rc_others), 're-checked string')} stand" if len(rc_others) else ""))
new2 = summary_rates(recheck["new_matcher_final"])
new2_judged = summary_rates(validation["round2"]["new_matcher_final"])
old2 = summary_rates(validation["round2"]["legacy"])
old12 = summary_rates(validation["legacy_both_rounds"])
new1 = summary_rates(validation["round1"]["new_matcher_v1"])
r1_n, r2_n = new1["judged"], new2["judged"]
rounds_text = (f"two rounds of {dp.fmt_int(r2_n)} strings each" if r1_n == r2_n
               else f"two rounds of {dp.fmt_int(r1_n)} and {dp.fmt_int(r2_n)} strings")
r2_p, r2_r, r2_p_ci, r2_r_ci = new2["precision"], new2["recall"], new2["precision_ci"], new2["recall_ci"]
r2_rw = new2["recall_w"]
r2_wrong, r2_missed, r2_missed_papers = new2["wrong"], new2["missed"], new2["missed_papers"]
old2_p, old2_r, old2_wrong = old2["precision"], old2["recall"], old2["wrong"]
old12_n, old12_p, old12_r = old12["judged"], old12["precision"], old12["recall"]
assert r2_missed > r2_wrong and r2_rw > r2_r, \
    "text: the remaining errors are mostly misses; the paper-weighted recall is higher than the per-string recall"
assert r2_r_ci[1] - r2_r_ci[0] > r2_p_ci[1] - r2_p_ci[0], "text: the intervals are wide, especially for recall"

# Paired comparison on the round-2 strings, which both matchers were judged on. "Right" =
# correct, or correctly left unmatched; uncertain verdicts are left out. Exact McNemar test.
r2_rows = judged[judged["round"] == "2"].copy()
# with the current dump: re-checked strings take their new verdict, and strings matched now are not unmatched
r2_rows["new_verdict"] = r2_rows["row_id"].map(recheck_rows["verdict_now"]).fillna(r2_rows["new_verdict"])
assert r2_rows["new_verdict"].value_counts().to_dict() == {
    k: v for k, v in recheck["new_matcher_final"]["counts"].items() if v}, "round2_recheck.csv and summary.json differ"


def sample_weighted(rows: pd.DataFrame, verdicts: pd.Series) -> dict:
    """Verdict counts weighted by n_papers: the papers in which each string occurred when its sample was drawn."""
    return rows["n_papers"].astype(int).groupby(verdicts).sum().to_dict()


# The paper weights in summary.json are those n_papers, not the paper counts of the current data.
r1_rows, r2_v166 = judged[judged["round"] == "1"], judged[judged["round"] == "2"]
assert all({k: v for k, v in entry["weighted_counts"].items() if v} == sample_weighted(rows, verdicts)
           for entry, rows, verdicts in (
               (recheck["new_matcher_final"], r2_rows, r2_rows["new_verdict"]),
               (validation["round2"]["new_matcher_final"], r2_v166, r2_v166["new_verdict"]),
               (validation["round2"]["legacy"], r2_v166, r2_v166["old_verdict"]),
               (validation["legacy_both_rounds"], judged, judged["old_verdict"]),
               (validation["round1"]["new_matcher_v1"], r1_rows, r1_rows["new_verdict"]))), \
    "text: the paper weights are the papers in which a string occurred when its sample was drawn"
r2_matched = (r2_rows["row_id"].map(recheck_rows["top_name_now"].notna())
              .fillna(r2_rows["new_method"].ne("none")).astype(bool))
RIGHT, NOT_RIGHT = {"correct", "correct_unmatched"}, {"wrong", "missed"}
n_new_better = int((r2_rows["new_verdict"].isin(RIGHT) & r2_rows["old_verdict"].isin(NOT_RIGHT)).sum())
n_old_better = int((r2_rows["old_verdict"].isin(RIGHT) & r2_rows["new_verdict"].isin(NOT_RIGHT)).sum())
n_disc = n_new_better + n_old_better
p_mcnemar = min(1.0, 2 * sum(math.comb(n_disc, i) for i in range(min(n_new_better, n_old_better) + 1))
                / 2 ** n_disc)
p_text = "< 0.001" if p_mcnemar < 0.001 else f"= {p_mcnemar:.3f}"

# Strings the final matcher left unmatched in round 2: no ROR record vs missed.
unmatched2 = r2_rows[~r2_matched]
n_none = len(unmatched2)
n_none_noror = int((unmatched2["new_verdict"] == "correct_unmatched").sum())

# Round-2 strings whose judged true organisation is not AUTH: what partner statistics rest on.
partner_rows = r2_rows[r2_rows["true_ror_id"].ne(config.AUTH_ROR)]
n_ext_strings = len(partner_rows)
ext_new = rates(partner_rows["new_verdict"].value_counts().to_dict())
ext_old = rates(partner_rows["old_verdict"].value_counts().to_dict())
assert ext_new["recall"] < r2_r, "text: strings outside AUTH are harder to match"

VALIDATION_COLUMNS = {   # column label -> rates of that matcher on that sample
    f"Final, R2, {rc_ror}": new2,
    "Final, R2, v1.66": new2_judged,
    "Legacy, R2": old2,
    "Legacy, R1+R2": old12,
    "First version, R1": new1,
}
VALIDATION_MEASURES = [  # (row label, key, formatter)
    ("Strings judged", "judged", dp.fmt_int),
    ("Correct", "correct", dp.fmt_int),
    ("Wrong", "wrong", dp.fmt_int),
    ("Missed", "missed", dp.fmt_int),
    ("No ROR record", "correct_unmatched", dp.fmt_int),
    ("Uncertain", "uncertain", dp.fmt_int),
    ("Precision", "precision", dp.fmt_pct),
    ("95% CI", "precision_ci", ci),
    ("Recall", "recall", dp.fmt_pct),
    ("95% CI", "recall_ci", ci),
    ("Precision, paper-weighted", "precision_w", dp.fmt_pct),
    ("Recall, paper-weighted", "recall_w", dp.fmt_pct),
]
val_table = pd.DataFrame(
    [{"measure": label, **{col: fmt(r[key]) for col, r in VALIDATION_COLUMNS.items()}}
     for label, key, fmt in VALIDATION_MEASURES])

# ===================================================================== definitions
n_dece_strings = int(aff["is_dece"].sum())
n_dece_pos, n_pos = int(authors["is_dece"].sum()), len(authors)
dece_aff_eids = set(pubs.loc[pubs["has_dece_affiliation"], "eid"])
dece_author_eids = set(authors.loc[authors["is_dece"], "eid"])
every_dece_record_has_author = dece_aff_eids <= dece_author_eids
dece_author_note = ("every record with a DECE affiliation has at least one" if every_dece_record_has_author
                    else f"{dp.fmt_int(len(dece_aff_eids - dece_author_eids))} records with a DECE "
                         "affiliation have none")
collab = pubs["collab_type"].value_counts().reindex(data.COLLAB_TYPE_ORDER, fill_value=0)
# International records whose foreign affiliations are all unmatched (country read from the text).
foreign = partner_occ[partner_occ["country_code"].notna()
                      & partner_occ["country_code"].ne(config.HOME_COUNTRY)]
n_intl_text_only = int((~foreign.groupby("eid")["ror_id_top"].agg(lambda s: s.notna().any())).sum())
n_industry = int(pubs["has_industry_partner"].sum())
# normalised citation score: reviews are pooled with articles (too few reviews a year). The years are
# those of Citation impact: from NCS_FROM to the last year with two full calendar years of citations.
NCS_FROM, NCS_TO = 2000, LAST_FULL - 2
reviews = pubs[pubs["year"].between(NCS_FROM, NCS_TO) & pubs["doc_type"].eq("Review")]
reviews_per_year = reviews.groupby("year").size().reindex(range(NCS_FROM, NCS_TO + 1), fill_value=0)
n_reviews, max_reviews_year = len(reviews), int(reviews_per_year.max())
assert reviews_per_year.min() < 2, "text: too few reviews for a yearly average of their own"
n_partial_articles = int(pubs.loc[pubs["is_partial_year"], "doc_type"].eq("Article").sum())
conference_share = pubs.groupby("year")["source_kind"].agg(lambda s: s.eq("conference").mean())
assert conference_share[PARTIAL] < conference_share[LAST_FULL], \
    "text: conference proceedings reach Scopus months after the event (fewer in the partial year)"

# Affiliation holders and the external-partner variants (listed under the tables)
HOLDER_COLUMNS = ["n_holders", "n_dece_holders", "held_only_by_dece", "holder_author_pos"]
EXTERNAL_COLUMNS = ["n_countries_external", "n_partner_orgs_external", "is_international_external",
                    "has_national_partner_external", "has_industry_partner_external", "collab_type_external",
                    "n_dece_only_affiliations"]
assert set(HOLDER_COLUMNS) <= set(pub_aff.columns) and set(EXTERNAL_COLUMNS) <= set(pubs.columns), \
    "text: the holder and external-partner columns listed under the tables"
has_holder = usable["n_holders"].gt(0)                  # held_only_by_dece is empty exactly when no holder
n_no_holder, n_no_holder_papers = int((~has_holder).sum()), usable.loc[~has_holder, "eid"].nunique()
if n_no_holder == 0:
    no_holder_text = (f"All {dp.fmt_int(len(usable))} affiliation occurrences (society memberships aside) have at "
                      "least one holder, so the rule that an affiliation without a holder counts as external "
                      "decides nothing in this export")
else:
    no_holder_text = (f"{dp.fmt_int(n_no_holder)} of the {dp.fmt_int(len(usable))} affiliation occurrences (society "
                      f"memberships aside), on {n_of(n_no_holder_papers, 'record')}, have no holder and count as "
                      "external")
# DECE-only: an affiliation outside AUTH that only DECE authors hold (their own second affiliation).
# Small groups (config.MIN_DECE_HOLDERS), the site's rule: no number of papers that link to something only
# through DECE-only affiliations (an institution, a country, a region, a sector, a collaboration type, or all of
# them together, as in "international only through DECE-only affiliations"), in any year, period or subgroup and
# for totals too, may be shown or follow from the figures shown, unless it is zero or rests on at least
# MIN_HOLDERS distinct DECE authors who hold these affiliations. "Follows" covers an address-based value minus
# its external-partner value, a total minus the parts shown and a share times its denominator, also with the
# figures of other pages. The text names the pages that apply the rule (they leave such numbers out, with enough
# other values that they cannot be recovered by subtraction, or describe them in words). The figures of this page
# are checked below (SMALL_GROUPS), and its values for all countries together must be among those that the site
# publishes (BOTH_BASES).
MIN_HOLDERS = config.MIN_DECE_HOLDERS
dece_only = ~usable["is_auth"] & usable["held_only_by_dece"].fillna(False).astype(bool)
n_dece_only, n_dece_only_papers = int(dece_only.sum()), usable.loc[dece_only, "eid"].nunique()
dece_only_cc = usable.loc[dece_only, "country_code"]
n_dece_only_abroad = int((dece_only_cc.notna() & dece_only_cc.ne(config.HOME_COUNTRY)).sum())
n_dece_only_greece = int(dece_only_cc.eq(config.HOME_COUNTRY).sum())
n_dece_only_nocountry = int(dece_only_cc.isna().sum())
share_dece_only, share_dece_only_ext = n_dece_only / len(usable), n_dece_only / len(partner_occ)
# Collaboration types on the external-partner basis, and the records that change type
collab_ext = pubs["collab_type_external"].value_counts().reindex(data.COLLAB_TYPE_ORDER, fill_value=0)
moves = (pd.crosstab(pubs["collab_type"], pubs["collab_type_external"])
         .reindex(index=data.COLLAB_TYPE_ORDER, columns=data.COLLAB_TYPE_ORDER, fill_value=0))
n_intl_dece_only = int((pubs["is_international"] & ~pubs["is_international_external"]).sum())
n_intl_to_single = int(moves.at["international", "single-institution"])
n_intl_to_national = int(moves.at["international", "national"])
n_national_to_single = int(moves.at["national", "single-institution"])
recent_pubs = pubs[pubs["year"].between(*RECENT)]
intl_recent, intl_recent_ext = recent_pubs["is_international"].mean(), recent_pubs["is_international_external"].mean()
n_industry_ext = int(pubs["has_industry_partner_external"].sum())
# partner institutions named by at least one affiliation that is not DECE-only
n_partner_orgs_ext = int((~dece_only[partner_tops.index]).groupby(partner_tops).any().sum())
n_partner_orgs_dece_only = n_partner_orgs - n_partner_orgs_ext
assert np.triu(moves.to_numpy(), 1).sum() == 0 and n_intl_dece_only == n_intl_to_single + n_intl_to_national, \
    "text: on the external-partner basis records only move to a narrower collaboration type"

# Small groups on this page. SMALL_GROUPS lists every figure above that counts DECE-only affiliations, or the
# records or institutions that link to something only through them, and every such number that follows from
# the figures shown: an address-based value minus its external-partner value (international, single-
# institution and industry-partner records, partner institutions) and all years minus RECENT (the two shares
# of RECENT give its count, since the completeness chart gives its number of records). Each comes with the
# DECE-only occurrences it rests on; the other differences between the two collaboration splits are sums or
# differences of these numbers. held: one row per DECE-only occurrence (row: its index in usable) and DECE
# author who holds it; author ids are only counted, never shown.
held = (usable.loc[dece_only, ["eid", "holder_author_pos"]].rename_axis("row").reset_index()
        .assign(author_pos=lambda d: d["holder_author_pos"].str.split(";")).explode("author_pos")
        .astype({"author_pos": "int64"})
        .merge(authors[["eid", "author_pos", "author_id", "is_dece"]], on=["eid", "author_pos"], how="left",
               validate="many_to_one"))
assert held["is_dece"].eq(True).all(), "every holder of a DECE-only affiliation is a DECE author"


def rests_on(occurrences: pd.Series) -> int:
    """Distinct DECE authors who hold the DECE-only occurrences among *occurrences* (a mask over usable)."""
    return held.loc[held["row"].isin(usable.index[occurrences & dece_only]), "author_id"].nunique()


def in_records(records: pd.Series) -> pd.Series:
    """Mask over usable: the occurrences in the records that *records* (a mask over pubs) selects."""
    return usable["eid"].isin(pubs.loc[records, "eid"])


aff_cc = usable["country_code"]                       # DECE-only occurrences are outside AUTH
abroad = aff_cc.notna() & aff_cc.ne(config.HOME_COUNTRY)
partner_org = usable["ror_id_top"].where(~usable["is_auth"] & usable["ror_id_top"].ne(config.AUTH_ROR))
# a national partner affiliation is matched to a partner institution, or unmatched with a country and naming an
# institution, which data/processed does not record: the matched ones give a lower bound of the holders
matched_partner = partner_org.notna()
# industry partner as in the pipeline: a partner organisation, or its top level, of ROR type company
company = set(orgs.loc[orgs["types"].fillna("").str.split(";").map(lambda t: "company" in t), "ror_id"])
industry = matched_partner & (usable["ror_id"].isin(company) | partner_org.isin(company))
for flag, occurrences in (("has_industry_partner", industry),
                          ("has_industry_partner_external", industry & ~dece_only)):
    assert pubs["eid"].isin(usable.loc[occurrences, "eid"]).eq(pubs[flag]).all(), f"{flag}: industry rule out of sync"
intl_only = pubs["is_international"] & ~pubs["is_international_external"]
intl_to_single = intl_only & pubs["collab_type_external"].eq("single-institution")
intl_to_national = intl_only & pubs["collab_type_external"].eq("national")
national_to_single = pubs["collab_type"].eq("national") & pubs["collab_type_external"].eq("single-institution")
industry_only = pubs["has_industry_partner"] & ~pubs["has_industry_partner_external"]
in_recent = pubs["year"].between(*RECENT)
assert usable.loc[dece_only & abroad & in_records(intl_only), "eid"].nunique() == n_intl_dece_only, \
    "each record international only through DECE-only affiliations has one abroad"
orgs_dece_only = set(partner_tops) - set(partner_tops[~dece_only[partner_tops.index]])
assert len(orgs_dece_only) == n_partner_orgs_dece_only, "partner institutions named only in DECE-only affiliations"
SMALL_GROUPS = {   # number shown, or following from numbers shown -> (its value, the DECE-only occurrences behind it)
    "DECE-only occurrences": (n_dece_only, dece_only),
    "records with DECE-only affiliations": (n_dece_only_papers, dece_only),
    "DECE-only occurrences abroad": (n_dece_only_abroad, abroad),
    "DECE-only occurrences in Greece": (n_dece_only_greece, aff_cc.eq(config.HOME_COUNTRY)),
    "DECE-only occurrences without a country": (n_dece_only_nocountry, aff_cc.isna()),
    "records international only through them": (n_intl_dece_only, abroad & in_records(intl_only)),
    "of these, single-institution on the external-partner basis": (
        n_intl_to_single, abroad & in_records(intl_to_single)),
    "of these, national on the external-partner basis": (n_intl_to_national, abroad & in_records(intl_to_national)),
    "national records that become single-institution": (
        n_national_to_single, matched_partner & in_records(national_to_single)),
    "single-institution records added on the external-partner basis": (
        int(collab_ext["single-institution"] - collab["single-institution"]),
        (abroad & in_records(intl_to_single)) | (matched_partner & in_records(national_to_single))),
    "records of RECENT international only through them": (
        int((intl_only & in_recent).sum()), abroad & in_records(intl_only & in_recent)),
    "records of the other years international only through them": (
        int((intl_only & ~in_recent).sum()), abroad & in_records(intl_only & ~in_recent)),
    "records with an industry partner only through them": (
        n_industry - n_industry_ext, industry & in_records(industry_only)),
    "partner institutions named only in them": (n_partner_orgs_dece_only, partner_org.isin(orgs_dece_only)),
}
assert SMALL_GROUPS["records of RECENT international only through them"][0] == round(
    (intl_recent - intl_recent_ext) * len(recent_pubs)), "small groups: the RECENT count behind the two shares"
too_small = [name for name, (n, occurrences) in SMALL_GROUPS.items() if n and rests_on(occurrences) < MIN_HOLDERS]
assert not too_small, (f"small groups: fewer than {MIN_HOLDERS} DECE authors hold the DECE-only affiliations behind "
                       f"{too_small}; leave these numbers, and those from which they follow, out of the text")
# The values for all countries together on both bases: how many records change collaboration type between the two
# bases (the difference of each type, address-based minus external-partner count, and the records of each change).
# Other pages give them too, by year, by period and for spans of years, and readers can combine the values of all
# pages, knowing also that a value left out stands for at least one record, so the site chooses them once
# (dece_stats.small_groups). This page gives them for all years (the two splits and the changes) and for RECENT
# (its two shares; the completeness chart gives its records): each must be one of the values the site publishes,
# with the same number. Their differences (all years minus RECENT) are covered by the site's check of all its
# values together.
rule = sg.site()
ALL_YEARS, RECENT_YEARS = range(year_min, PARTIAL + 1), range(RECENT[0], RECENT[1] + 1)
BOTH_BASES = {   # value given on both bases -> (its years, kind: a type's difference or a change, value on this page)
    **{f"{t} records": (ALL_YEARS, t, int(collab[t] - collab_ext[t])) for t in data.COLLAB_TYPE_ORDER},
    "records international only through DECE-only affiliations": (ALL_YEARS, sg.INTL, n_intl_dece_only),
    "of these, single-institution on the external-partner basis": (ALL_YEARS, sg.TO_SINGLE, n_intl_to_single),
    "of these, national on the external-partner basis": (ALL_YEARS, sg.TO_NAT, n_intl_to_national),
    "national records that become single-institution": (ALL_YEARS, sg.NAT_SINGLE, n_national_to_single),
    "records of RECENT international only through them (two shares)": (
        RECENT_YEARS, sg.INTL, SMALL_GROUPS["records of RECENT international only through them"][0]),
}
unpublished = [name for name, (years, kind, _) in BOTH_BASES.items() if not rule.is_published(years, kind)]
assert not unpublished, (f"small groups: {unpublished} are not among the values for all countries together that the "
                         "site publishes (dece_stats.small_groups); leave them out of the text")
assert all(rule.value(years, kind) == value for years, kind, value in BOTH_BASES.values()), \
    "small groups: the values for all countries together differ from those of dece_stats.small_groups"
failed_rule = [check for check, ok in rule.checks().items() if not ok]
assert not failed_rule, f"small groups: the site's rule fails {failed_rule}"

# ===================================================================== data quality
years = pubs["year"]
is_recent = years.between(*RECENT)
conference = pubs["source_kind"].eq("conference")
doi_share = pubs["doi"].notna().mean()
doi_recent = pubs.loc[is_recent, "doi"].notna().mean()


def quality_row(field: str, present: pd.Series, when: pd.Series, used_on: str) -> dict:
    """Completeness of one field: share of items with a value, over all years and in RECENT."""
    present = present.astype(bool)
    return {"field": field,
            "items": len(present),
            "share": float(present.mean()),
            "recent": float(present[when.between(*RECENT)].mean()),
            "used_on": used_on}


# "Used on": the other pages that read the field or a field derived from it (the country gives the international
# flag; the ROR match outside AUTH the partner institutions, the sectors and the national and single-institution
# types; the holders the external-partner fields; the DOI and the conference name are fallbacks of the publisher
# groups), the pages that rest most on it first. Kept by hand: check it when a page starts to read another field.
quality = pd.DataFrame([
    quality_row("DOI", pubs["doi"].notna(), years,
                "Citation impact; Venues & publishers; Funding & open access; Overview"),
    quality_row("Publisher", pubs["publisher"].notna(), years, "Venues & publishers; Funding & open access; Overview"),
    quality_row("Conference name (conference papers)", pubs.loc[conference, "conference_name"].notna(),
                years[conference], "Venues & publishers; Funding & open access; Overview"),
    quality_row("Funding details", pubs["has_funding"], years, "Funding & open access; Citation impact; Overview"),
    quality_row("Open Access field", pubs["oa_any"], years,
                "Funding & open access; Citation impact; Venues & publishers; Research topics; Overview"),
    quality_row("Correspondence address with an affiliation", pubs["dece_corresponding"].notna(), years,
                "Research community"),
    quality_row("Scopus author id (author positions)", authors["author_id"].notna(), authors["year"],
                "Research community; Overview; small-group checks of Citation impact, Collaboration and Global reach"),
    quality_row("Country (affiliation occurrences)", usable["country_code"].notna(), usable["year"],
                "Collaboration; Global reach; every other page but Venues & publishers"),
    quality_row("Holder found (affiliation occurrences)", has_holder, usable["year"],
                "Collaboration; Global reach; Citation impact; Funding & open access; Overview"),
    quality_row("ROR match (affiliation occurrences outside AUTH)", partner_occ["matched"], partner_occ["year"],
                "Collaboration; Global reach; Citation impact; Research output; Research community; Overview"),
])
# The rows of affiliation occurrences leave out the society memberships (caption): their items are those of
# usable, all of them or those outside AUTH, and usable holds every occurrence but the society memberships.
occ_rows = quality["field"].str.contains("affiliation occurrences", regex=False)
assert (occ_rows.any() and quality.loc[occ_rows, "items"].isin([len(usable), len(partner_occ)]).all()
        and len(usable) == n_occ - int(mm["methods_occurrences"]["ignored"])), \
    "text: the rows of affiliation occurrences leave out the society memberships (tbl-method-quality caption)"

COMPLETENESS = {   # heatmap rows: share of records with a value
    "DOI": pubs["doi"].notna(),
    "Publisher": pubs["publisher"].notna(),
    "Funding details": pubs["has_funding"],
    "Correspondence address": pubs["dece_corresponding"].notna(),
}
pub_period = years.map(chart_period)
completeness = pd.DataFrame(COMPLETENESS).astype(float).groupby(pub_period).mean().reindex(PERIODS).T
period_n = pub_period.value_counts().reindex(PERIODS)
assert period_n[early_label] < period_n[second_period], "text: the early years are merged (few records)"

# Publisher: first year from which it is recorded for >= 90% of the records of every later full year.
pub_by_year = pubs[years < PARTIAL].groupby("year")["publisher"].agg(lambda s: s.notna().mean())
publisher_from = int(next(y for y in pub_by_year.index if (pub_by_year.loc[y:] >= 0.9).all()))
publisher_pre = pubs.loc[years < publisher_from, "publisher"].notna().mean()
publisher_post = pubs.loc[years.between(publisher_from, LAST_FULL), "publisher"].notna().mean()
funding_earlier = pubs.loc[years.between(*EARLIER), "has_funding"].mean()
funding_recent = pubs.loc[is_recent, "has_funding"].mean()
assert publisher_pre < publisher_post and funding_earlier < funding_recent, \
    "text: several fields are less complete in older records; funding acknowledgements rose"
# Correspondence address: the least complete of the periods since MODERN_FROM
has_corr = pubs["dece_corresponding"].notna()
corr_by_period = has_corr.groupby(years.map(periods.period_of)).mean().reindex(modern)
P_LOW = corr_by_period.idxmin()
corr_all, corr_low = has_corr.mean(), corr_by_period[P_LOW]
assert corr_low < corr_all, "text: available for corr_all of all records but for only corr_low of P_LOW"
n_in_press = int(pubs["pub_stage"].eq("Article in press").sum())
n_editorial = int(pubs["doc_type"].eq("Editorial").sum())
n_erratum = int(pubs["doc_type"].eq("Erratum").sum())

# Query precision. A DECE affiliation string has elec* and aristot* words in the same string.
n_dece_aff = int(pubs["has_dece_affiliation"].sum())
share_dece_aff = n_dece_aff / n_pubs
no_dece = pubs.loc[~pubs["has_dece_affiliation"], ["eid", "year"]]
n_no_dece = len(no_dece)
no_dece_years = join_and(sorted(no_dece["year"]))
country_names = set(countries["country_name"].dropna())
no_dece_affs = pub_aff[pub_aff["eid"].isin(no_dece["eid"])].groupby("eid")["affiliation"].agg(set)
n_country_only = int(no_dece_affs.map(lambda s: s <= country_names).sum())
no_dece_note = ("give only a country as their affiliation" if n_country_only == n_no_dece
                else f"have no DECE string ({dp.fmt_int(n_country_only)} give only a country)")
assert n_country_only == n_no_dece, "text: the records without a DECE string give only a country (key findings)"
# The elec* rule is broader than the query words: keyword check for DECE strings of other AUTH
# units, i.e. strings naming Physics, Journalism or the medical school and no electrical or
# computer engineering department ("Section of Electr. and Computers, Dept. of Physics").
OTHER_UNIT = r"physic|journalism|dermatolog|school of medicine|medical school"
ECE_DEPT = (r"electrical|electr\.?\s*,?\s*eng|elec\.?\s*comp\.?\s*eng"
            r"|(?:electronics?|electrical)\s+(?:and|&)\s+comp\w*\.?\s*eng|computer\s+eng|\bECE\b")
dece_occ = pub_aff[pub_aff["is_dece"]].assign(
    other_unit=lambda d: d["affiliation"].str.contains(OTHER_UNIT, case=False)
    & ~d["affiliation"].str.contains(ECE_DEPT, case=False))
n_other_strings = int(dece_occ.loc[dece_occ["other_unit"], "affiliation"].nunique())
assert all(dece_occ.loc[dece_occ["other_unit"], "affiliation"].str.contains(p, case=False).any() for p in (
    r"electronics lab.*physic|physic.*electronics lab", r"section of elec.*physic", r"electronic media.*journalism",
    r"medical school|school of medicine")), "text: the examples of DECE strings of other AUTH units"
other_unit_rec = dece_occ.groupby("eid")["other_unit"].all()
other_unit_eids = other_unit_rec[other_unit_rec].index
n_other_unit = len(other_unit_eids)
assert dece_occ.loc[dece_occ["eid"].isin(other_unit_eids) & dece_occ["other_unit"], "affiliation"].str.contains(
    "physic", case=False).any(), "text: such as the Department of Physics (key findings)"
# Wording check: do these records name AUTH only through such other-unit strings?
other_unit_auth = pub_aff[pub_aff["eid"].isin(other_unit_eids) & pub_aff["is_auth"]]
other_unit_text = ("name AUTH only through other units of it"
                   if other_unit_auth["affiliation"].isin(dece_occ.loc[dece_occ["other_unit"], "affiliation"]).all()
                   else "match the query only through affiliations of other AUTH units")
n_confirmed = int(dece_occ.loc[~dece_occ["other_unit"], "eid"].nunique())
share_confirmed = n_confirmed / n_pubs
assert share_confirmed > 0.99, "text: the records kept are precise (key findings)"
n_doubtful = n_pubs - n_confirmed
share_doubtful = n_doubtful / n_pubs

Where do the numbers on this site come from, how were they produced, and how far can they be
trusted? This page describes the data sources and the processing pipeline, how affiliations were
linked to institutions and how accurate that linking is, the definitions used on every page, and the
known gaps in the data. All pages use the same {eval}`dp.fmt_int(n_pubs)` records dated
{eval}`str(year_min)` to {eval}`str(year_max)`, selected from a Scopus export taken on {eval}`as_of_text`.

In [ ]:
dp.kpi_html([
    dp.KPI("Scopus records", n_pubs, f"{year_min}–{year_max} ({PARTIAL} to {as_of_short})"),
    dp.KPI("Affiliation strings", n_strings, f"distinct; {dp.fmt_int(n_occ)} occurrences"),
    dp.KPI("Linked to ROR", dp.fmt_pct(occ_share),
           f"of {dp.fmt_int(occ_usable)} occurrences (society memberships aside); "
           f"{dp.fmt_pct(occ_share_ext)} outside AUTH"),
    dp.KPI("Matching precision", dp.fmt_pct(r2_p),
           f"recall {dp.fmt_pct(r2_r)}; fresh sample of {dp.fmt_int(r2_n)} strings"),
], min_width=140)

:::{important} Key findings
- Scopus applies the search query across the affiliations of a record: {eval}`dp.fmt_int(n_split)` of
  the {eval}`dp.fmt_int(n_exported)` records it returned ({eval}`dp.fmt_pct(share_split)`) hold the query
  words only in different affiliations and are left out. The records kept are precise:
  {eval}`dp.fmt_int(n_confirmed)` of {eval}`dp.fmt_int(n_pubs)` ({eval}`dp.fmt_pct(share_confirmed, 2)`)
  carry an affiliation of the department itself. Of the other {eval}`dp.fmt_int(n_doubtful)`, {eval}`dp.fmt_int(n_no_dece)` give only a country as their
  affiliation and {eval}`dp.fmt_int(n_other_unit)` {eval}`other_unit_text`, such as the Department of
  Physics.
- Leaving aside society memberships, {eval}`dp.fmt_pct(occ_share)` of the affiliation occurrences and
  {eval}`dp.fmt_pct(occ_share_ext)` of those outside AUTH are linked to an organisation in the Research
  Organization Registry (ROR). Outside AUTH the share drops to {eval}`dp.fmt_pct(ext_pre_modern)` in records
  published before {eval}`str(MODERN_FROM)`.
- On a fresh random sample of {eval}`dp.fmt_int(r2_n)` affiliation strings, the matcher's precision is
  {eval}`dp.fmt_pct(r2_p)` (95% CI {eval}`ci(r2_p_ci)`) and its recall {eval}`dp.fmt_pct(r2_r)`
  ({eval}`ci(r2_r_ci)`) with ROR {eval}`rc_ror`. The legacy matcher it replaces scored {eval}`dp.fmt_pct(old2_p)` and
  {eval}`dp.fmt_pct(old2_r)` on the same strings.
- Several fields are less complete in older records. The publisher is recorded for
  {eval}`dp.fmt_pct(publisher_pre)` of the records published before {eval}`str(publisher_from)` and for
  {eval}`dp.fmt_pct(publisher_post)` of those from {eval}`str(publisher_from)` to {eval}`str(LAST_FULL)`.
  Funding acknowledgements appear on {eval}`dp.fmt_pct(funding_earlier)` of the {eval}`P_EARLIER`
  records and on {eval}`dp.fmt_pct(funding_recent)` of the {eval}`P_RECENT` records, a rise that may
  reflect more funded research as well as more complete indexing.
- Of the {eval}`dp.fmt_int(len(usable))` affiliation occurrences (society memberships aside),
  {eval}`dp.fmt_int(n_dece_only)` ({eval}`dp.fmt_pct(share_dece_only)`, on
  {eval}`dp.fmt_int(n_dece_only_papers)` records) are affiliations outside AUTH held only by DECE
  authors, that is, their own second affiliations; {eval}`dp.fmt_int(n_dece_only_abroad)` of them are
  abroad. Counting only affiliations of co-authors who are not DECE authors,
  {eval}`dp.fmt_int(collab_ext['international'])` records are international instead of
  {eval}`dp.fmt_int(collab['international'])`, and {eval}`dp.fmt_pct(intl_recent_ext)` instead of
  {eval}`dp.fmt_pct(intl_recent)` of those of {eval}`P_RECENT`.
:::

(method-sources)=
## Data sources

The analysis combines three sources ([](#tbl-method-sources)). The Scopus records come from the
advanced-search query {eval}`query`, run on {eval}`as_of_text`. It asks for documents with an affiliation
that contains one of the words *elec*, *elect*, *electr* or *electrical* and one of *aristotle* or
*aristotelian*, but Scopus applies it across the affiliations of a record, so that a physics paper of
AUTH with a co-author in an electrical engineering department elsewhere matches too. The pipeline
therefore selects the records itself (`select_records` in `dece_stats.pipeline`): of the
{eval}`dp.fmt_int(n_exported)` records returned, {eval}`left_out_text`. The other
{eval}`dp.fmt_int(n_pubs)` are {eval}`kept_text`. {eval}`str(PARTIAL)` is covered only up to the export
date ({eval}`dp.fmt_int(n_partial)` records).

:::{table} Data sources and the snapshots used.
:label: tbl-method-sources
| Source | Snapshot | Used for |
|---|---|---|
| Scopus (Elsevier), CSV export of the query results | {eval}`as_of_text`: {eval}`dp.fmt_int(n_exported)` records, {eval}`dp.fmt_int(n_pubs)` of them kept ({eval}`str(year_min)`–{eval}`str(year_max)`) | Publications, citation counts, affiliations, author ids, funding, open access |
| Research Organization Registry (ROR), data dump | {eval}`ror_version` of {eval}`ror_date_text` | Institutions: names in all languages, acronyms, types, locations and parent organisations |
| Natural Earth, Admin 0 countries | 1:{eval}`ne_scale` scale | ISO codes, continents and UN regions of the {eval}`dp.fmt_int(n_countries)` countries in the data (small countries missing at this scale, such as Malta and Singapore, come from a UN M49 table built into the pipeline) |
:::

From each Scopus record the pipeline reads the EID, title, year, source title, document type,
publication stage, citation count (*Cited by*), DOI, ISSN, publisher, language, conference name,
open-access field, affiliations, authors with affiliations, author ids, funding details and
correspondence address. Author names are read only to split the per-author affiliation field and to
keep names out of the affiliation and funder entries; they are not stored. E-mail addresses are
discarded, and the correspondence address is reduced to one flag per record: whether a corresponding
author gives a DECE affiliation. No author name, e-mail address or author id appears on this site,
and no metric is shown for any individual. The pipeline does not keep the abstracts and keywords of
the export, so [Research topics](topics.ipynb) relies on titles alone.

Scopus content is subject to Elsevier's terms of use. The raw export and the per-paper tables stay in
the project's private repository; the site shows aggregates and the bibliographic details of individual
highly cited papers.

## Processing pipeline

The Python package `dece_stats` turns the inputs into the tables that every page reads
([](#fig-method-pipeline)).

:::{figure}
:label: fig-method-pipeline
```{mermaid}
flowchart TD
  subgraph inputs [Inputs]
    direction LR
    scopus[("Scopus export")]
    ror[("ROR data dump")]
    overrides[("Manual overrides")]
    ne[("Natural Earth")]
  end
  scopus --> parse["Load, select and parse the records"]
  parse --> match["Match affiliations to ROR and roll up"]
  ror --> match
  overrides --> match
  match --> derive["Derive paper-level fields"]
  parse --> derive
  ne --> derive
  derive --> processed[("data/processed")]
  processed --> pages["Site pages, executed by MyST"]
  validation[("Matching validation")] --> pages
```
From the inputs to the site. Cylinders are data sets, boxes are steps of the `dece_stats` package.
:::

1. **Load and select** (`dece_stats.scopus`, `dece_stats.pipeline`): read the export as text, drop
   duplicate EIDs and select the records (see [Data sources](#method-sources)). Text is cleaned of HTML
   entities and e-mail addresses as it is parsed.
2. **Parse**: split each record's affiliation field into distinct affiliation strings, the per-author
   affiliation field into one chunk per author (linked to the author ids by position), the funding
   field into funder entries and the open-access field into routes; flag DECE affiliations and DECE
   authors, and find the authors who hold each affiliation string (see [Definitions](#method-definitions)).
3. **Match** (`dece_stats.matching`, `dece_stats.ror`): match every distinct affiliation string to ROR
   and roll the match up to its top-level organisation (see [Affiliation matching](#method-matching)).
4. **Derive** (`dece_stats.pipeline`): per record, the countries, partner organisations, collaboration
   type, industry partners and the DECE first-, last- and corresponding-author flags. The country and
   partner fields are derived twice: from all affiliations (address-based) and without the affiliations
   outside AUTH that only DECE authors hold (external-partner).
5. **Write** the tables listed in [](#tbl-method-tables), and `metadata.json` (versions, dates,
   record counts, matching and holder summaries), to `data/processed`, after checking every table again for e-mail addresses. A rerun on
   the same inputs rewrites identical tables (only the timestamp in `metadata.json` changes); the
   tables used here were generated on {eval}`generated_text`.

To reproduce the site from the project repository, with the Scopus export and the ROR dump in
`data/raw`:

```bash
pip install -e ".[pipeline,site]"
python -m dece_stats.pipeline            # data/raw -> data/processed
cd site && myst build --html --execute   # runs every page notebook
```

In [ ]:
#| label: cell-method-tables
dp.table_html(
    tables,
    columns={"file": "File", "rows": "Rows", "row": "One row per"},
)

:::{figure} #cell-method-tables
:kind: table
:label: tbl-method-tables
Tables written by the pipeline to `data/processed`. The site reads only these tables and the
matching validation in `data/validation`.
:::

The holders of each affiliation (see [Definitions](#method-definitions)) are in four columns of
`publication_affiliations.csv`: `n_holders`, `n_dece_holders`, `held_only_by_dece` (empty when no
author holds the string) and `holder_author_pos`, the positions of the holders in the author list, which
the pages use only to count how many DECE authors a number rests on. `publications.csv` adds the
external-partner variants of the collaboration fields (`n_countries_external`, `n_partner_orgs_external`, `is_international_external`,
`has_national_partner_external`, `has_industry_partner_external` and `collab_type_external`) and
`n_dece_only_affiliations`, the number of DECE-only affiliations of the record.

(method-matching)=
## Affiliation matching

Partner, sector and country statistics depend on knowing which institution each affiliation names.
Scopus gives affiliations as free text, written differently from paper to paper, so the
{eval}`dp.fmt_int(n_strings)` distinct affiliation strings of the data set
({eval}`dp.fmt_int(n_occ)` occurrences in records) were matched to ROR by a rule-based matcher that
prefers leaving a string unmatched to matching it wrongly:

1. **Country first.** The country is parsed from the end of the string (with aliases such as USA, UK,
   Korea or Türkiye; Hong Kong, Macau and Taiwan are kept separate), independently of the
   organisation. The matched organisation's country is used instead only when the string names no
   country, or when the places in the string point to the organisation's country.
2. **Parts.** The string is split at commas. Postcodes, street addresses, place names and the country
   are set aside, and the remaining parts are ranked by how much they look like an organisation:
   universities first, then institutes, hospitals and companies, and department or laboratory names
   last.
3. **Exact match** of a normalised part (accents, abbreviations, word order, organisation words in
   other languages) against every ROR name, alias and label, preferring organisations in the parsed
   country.
4. **Acronym match**, only within the parsed country and only when the acronym is unique there.
5. **Fuzzy match** within the parsed country, requiring a shared distinctive word: a rapidfuzz
   token-sort similarity of at least {eval}`cut` ({eval}`cut_neutral` for parts without an organisation
   word, {eval}`cut_nocountry` for strings whose country could not be parsed), or a part that differs
   from a single organisation's name only by place names or generic words. Department-level parts are
   never fuzzy-matched, and fuzzy or acronym candidates that are universities or hospitals far from
   every place named in the string are rejected.
6. **Ignore and override.** Strings that only state a society membership (such as IEEE or the
   Technical Chamber of Greece) are ignored. Manual overrides, applied last, set the match of
   {eval}`n_of(n_override, 'string')`.
7. **Roll-up** through ROR's parent links to the top-level organisation, so that a research institute
   counts with its parent organisation. Universities stay their own top level and are not merged into
   university systems or federations.

In [ ]:
#| label: cell-method-match
dp.table_html(
    methods,
    columns={"outcome": "Outcome", "strings": "Distinct strings", "occurrences": "Occurrences",
             "share": "Share of occurrences"},
    formats={"share": "pct"},
)

:::{figure} #cell-method-match
:kind: table
:label: tbl-method-match
Outcome of the affiliation matching, per distinct string and per occurrence in a record. The shares
leave out the strings that only state a society membership.
:::

Of the {eval}`dp.fmt_int(n_strings)` strings, {eval}`dp.fmt_int(n_ignored)` only state a society
membership and are left out ([](#tbl-method-match)). Of the other {eval}`dp.fmt_int(strings_usable)`,
{eval}`dp.fmt_int(strings_matched)` ({eval}`dp.fmt_pct(str_share)`) were matched. Counted by
occurrence the share is higher, {eval}`dp.fmt_int(occ_matched)` of {eval}`dp.fmt_int(occ_usable)`
({eval}`dp.fmt_pct(occ_share)`), because frequently used strings are matched more often than rare ones.
Coverage is lower outside AUTH, which is what the partner statistics rely on:
{eval}`dp.fmt_pct(str_share_ext)` of those strings and {eval}`dp.fmt_pct(occ_share_ext)` of their
occurrences. The matches identify {eval}`dp.fmt_int(n_partner_orgs)` distinct top-level partner
institutions besides AUTH.

In [ ]:
#| label: cell-method-coverage
x = list(range(len(PERIODS)))
fig = go.Figure()
for name, cov, color in (("Outside AUTH (partner affiliations)", cov_ext, dp.CATEGORICAL[0]),
                         ("All affiliations", cov_all, dp.INK["muted"])):
    fig.add_trace(go.Scatter(
        x=x, y=cov["share"], name=name, mode="lines+markers",
        line=dict(color=color, width=2.5), marker=dict(color=color, size=7),
        customdata=[[p, int(k), int(n)] for p, k, n in zip(PERIODS, cov["k"], cov["n"])],
        hovertemplate=(f"<b>%{{customdata[0]}}</b><br>{name}<br>%{{y:.1%}} matched "
                       "(%{customdata[1]:,} of %{customdata[2]:,} occurrences)<extra></extra>"),
    ))
low = math.floor(min(cov_all["share"].min(), cov_ext["share"].min()) * 10) / 10
fig.update_xaxes(tickvals=x, ticktext=[p.replace("–", "–<br>").replace(" (", "<br>(") for p in PERIODS],
                 range=[-0.5, len(x) - 0.5], showgrid=False, zeroline=False)
fig.update_yaxes(range=[low, 1.0], tickformat=".0%", title_text="Occurrences matched to ROR")
fig.update_layout(hovermode="closest", margin=dict(t=30))
dp.style_partial_year(fig, x[-1], label=f"{PARTIAL}: partial (to {as_of_short})")
fig.show()

:::{figure} #cell-method-coverage
:label: fig-method-coverage
Share of affiliation occurrences matched to a ROR organisation, by publication period, for the
affiliations outside AUTH and for all affiliations. Years before {eval}`str(EARLY_END + 1)` are merged
(few records); {eval}`str(PARTIAL)` is partial (to {eval}`as_of_short`).
:::

In every five-year period from {eval}`modern[0]` to {eval}`P_RECENT`, between {eval}`dp.fmt_pct(ext_min)`
and {eval}`dp.fmt_pct(ext_max)` of the affiliation occurrences outside AUTH were matched
([](#fig-method-coverage)), against {eval}`dp.fmt_pct(ext_second)` in {eval}`second_period` and
{eval}`dp.fmt_pct(ext_early)` of the {eval}`dp.fmt_int(n_ext_early)` occurrences before
{eval}`str(EARLY_END + 1)`. Partner counts for the older periods are therefore less complete. An unmatched string is not always a failure
of the matcher: of the {eval}`dp.fmt_int(n_none)` strings that the final matcher left unmatched in the
validation sample (next section), {eval}`dp.fmt_int(n_none_noror)` name an organisation that has no
ROR record, such as several Greek companies and public bodies, or no organisation at all.

(method-validation)=
## Validation of the matching

The matcher was checked on random samples of distinct affiliation strings, in {eval}`rounds_text`.
For every string an AI assistant (Claude agents) identified the true organisation, looked it up in the
ROR dump then in use, v1.66 (names, aliases, acronyms, relations and status), and compared it with the top-level
organisation returned by each matcher. Round 1 measured a first version of the new matcher, and its
errors were used to improve it, so it is not an independent test. Round 2 is a fresh sample drawn after
those fixes and gives the headline estimate. The legacy matcher, which fuzzy-matched every
comma-separated part of a string against the ROR names of all countries and which the new one
replaces, was judged on both samples.

With ROR {eval}`rc_ror` of {eval}`rc_date`, the final matcher gives a different result for
{eval}`n_of(len(rc_changed), 'round-2 string')`: it now {eval}`rc_moves_text`. The changed strings were
judged again against the new dump, and the strings judged missed or without a ROR record were
checked for records added or removed since ({eval}`dp.fmt_int(rc_n)` strings in all, listed in
`data/validation/round2_recheck.csv`): {eval}`rc_text`. The round-2 figures of the final matcher below are those with the current dump.

Each string received one verdict per matcher, judged at the level of the top-level organisation:

* **correct**: the right organisation, or a ROR parent or successor of it;
* **wrong**: a different organisation;
* **missed**: no match, although ROR has a suitable record;
* **no ROR record**: no match, and ROR has no suitable record (this includes strings that name no
  organisation);
* **uncertain**: the judge could not decide; left out of the metrics.

*Precision* = correct / (correct + wrong) is how often a returned organisation is right. *Recall* =
correct / (correct + wrong + missed) is the share of matchable strings that end up with the right
organisation. The 95% confidence intervals are Wilson score intervals. The paper-weighted variants
weight each string by the number of papers in which it occurred when its sample was drawn.

In [ ]:
#| label: cell-method-validation
ROWS = [  # (label, rates, colour)
    (f"Final matcher<br>round 2, ROR {rc_ror}", new2, dp.CATEGORICAL[0]),
    ("Final matcher<br>round 2, ROR v1.66", new2_judged, dp.CATEGORICAL[0]),
    ("Legacy matcher<br>round 2 (same strings)", old2, dp.CATEGORICAL[1]),
    ("Legacy matcher<br>rounds 1 and 2", old12, dp.CATEGORICAL[1]),
    ("First version<br>round 1 (tuning sample)", new1, dp.INK["muted"]),
]
METRICS = [("precision", "n_p", "matched"), ("recall", "n_r", "matchable")]
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.08)
for col, (metric, nkey, noun) in enumerate(METRICS, start=1):
    for label, r, color in ROWS:
        value, (lo, hi), weighted = r[metric], r[f"{metric}_ci"], r[f"{metric}_w"]
        name = label.replace("<br>", ", ")
        fig.add_trace(go.Scatter(
            x=[value], y=[label], mode="markers", showlegend=False,
            marker=dict(color=color, size=10),
            error_x=dict(type="data", symmetric=False, array=[hi - value], arrayminus=[value - lo],
                         color=color, thickness=2, width=6),
            hovertemplate=(f"<b>{name}</b><br>{metric.capitalize()} {value:.1%} "
                           f"(95% CI {lo:.1%}–{hi:.1%})<br>{r['correct']:,} correct of "
                           f"{r[nkey]:,} {noun} strings<extra></extra>"),
        ), row=1, col=col)
        fig.add_trace(go.Scatter(
            x=[weighted], y=[label], mode="markers", showlegend=False,
            marker=dict(symbol="diamond-open", color=color, size=11, line=dict(width=2, color=color)),
            hovertemplate=f"<b>{name}</b><br>{metric.capitalize()} weighted by papers {weighted:.1%}<extra></extra>",
        ), row=1, col=col)
for symbol, legend_name in (("circle", "Per string, with 95% CI"), ("diamond-open", "Weighted by papers")):
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers", name=legend_name, hoverinfo="skip",
        marker=dict(symbol=symbol, color=dp.INK["muted"], size=10, line=dict(width=2, color=dp.INK["muted"])),
    ))
lowest = min(r[f"{m}_ci"][0] for _, r, _ in ROWS for m, _, _ in METRICS)
fig.update_xaxes(range=[math.floor(lowest * 20) / 20, 1.01], tickformat=".0%", dtick=0.05,
                 showgrid=True, zeroline=False)
fig.update_xaxes(title_text="Precision", row=1, col=1)
fig.update_xaxes(title_text="Recall", row=1, col=2)
fig.update_yaxes(autorange="reversed", showgrid=False)
fig.update_layout(height=390, hovermode="closest", margin=dict(t=36))
fig.show()

:::{figure} #cell-method-validation
:label: fig-method-validation
Precision and recall of the affiliation matchers on judged random samples of distinct affiliation
strings. Dots show the per-string estimate with its 95% Wilson interval, open diamonds the estimate
weighted by the number of papers in which each string occurred when its sample was drawn. Round 2
is the headline test of the final matcher, judged with ROR v1.66 and re-checked with {eval}`rc_ror`.
:::

In the fresh round-2 sample the final matcher returned a wrong organisation for
{eval}`n_of(r2_wrong, 'string')` and missed {eval}`n_of(r2_missed, 'matchable string')`; on the same
strings the legacy matcher returned {eval}`dp.fmt_int(old2_wrong)` wrong organisations
([](#fig-method-validation)). Compared string by string, the final matcher was right (a correct
match, or no match where ROR has no suitable record) where the legacy matcher was wrong or missed in
{eval}`dp.fmt_int(n_new_better)` cases, and the reverse happened in {eval}`dp.fmt_int(n_old_better)`
(exact McNemar test, p {eval}`p_text`). The remaining
errors of the final matcher are mostly misses, as intended: the {eval}`dp.fmt_int(r2_missed)` missed
strings occurred in only {eval}`dp.fmt_int(r2_missed_papers)` papers when the sample was drawn,
which is why the paper-weighted recall ({eval}`dp.fmt_pct(r2_rw)`) is higher than the per-string
recall. Pooled over both rounds
({eval}`dp.fmt_int(old12_n)` strings), the legacy matcher reaches a precision of
{eval}`dp.fmt_pct(old12_p)` and a recall of {eval}`dp.fmt_pct(old12_r)`.

Strings outside AUTH, on which the partner statistics rest, are harder to match. On the
{eval}`dp.fmt_int(n_ext_strings)` round-2 strings whose true organisation is not AUTH, the final
matcher's precision was {eval}`dp.fmt_pct(ext_new['precision'])` (95% CI
{eval}`ci(ext_new['precision_ci'])`) and its recall {eval}`dp.fmt_pct(ext_new['recall'])`
({eval}`ci(ext_new['recall_ci'])`), against {eval}`dp.fmt_pct(ext_old['precision'])` and
{eval}`dp.fmt_pct(ext_old['recall'])` for the legacy matcher.

In [ ]:
#| label: cell-method-validation-table
dp.table_html(
    val_table,
    columns={"measure": "", **{col: col for col in VALIDATION_COLUMNS}},
    numeric=list(VALIDATION_COLUMNS),
)

:::{figure} #cell-method-validation-table
:kind: table
:label: tbl-method-validation
Verdicts and metrics per matcher and sample (R1, R2: validation rounds 1 and 2; v1.66,
{eval}`rc_ror`: the ROR dump of the verdicts), computed from the
verdict counts in `data/validation/summary.json`. Precision and recall are per string, with 95% Wilson
intervals; the paper-weighted rows weight each string by the number of papers in which it
occurred when its sample was drawn.
:::

[](#tbl-method-validation) gives the verdict counts behind [](#fig-method-validation). The verdicts come
with caveats. They were made by AI agents with ROR look-ups and were not all checked
by hand, so a few may be wrong; a note records the reasoning behind every non-trivial verdict so that
they can be audited. With {eval}`dp.fmt_int(r2_n)` strings per round the intervals are wide, especially
for recall. Only the organisation was judged, not the country assigned to the string. ROR itself has
gaps: some organisations have no record, and some records lack the parent link that would roll an
institute up to its parent organisation (CERTH's Information Technologies Institute, for example, has no
link to CERTH), which undercounts the parent unless an override is added.

(method-definitions)=
## Definitions

Publication, paper and record
: One Scopus record of any document type; the pages use the three words interchangeably.

DECE affiliation
: An affiliation string with a word starting with *elec* and a word starting with *aristot*, ignoring
  case and accents. This mirrors the search query; {eval}`dp.fmt_int(n_dece_strings)` of the
  {eval}`dp.fmt_int(n_strings)` distinct strings qualify.

DECE author
: An author whose own affiliation text contains a DECE affiliation string. Scopus sometimes splits one
  affiliation into an AUTH string and a department string, so an *elec…* unit that names no
  institution, listed next to an *aristot…* string by the same author, also counts.
  {eval}`dp.fmt_int(n_dece_pos)` of the {eval}`dp.fmt_int(n_pos)` author positions are DECE authors,
  and {eval}`dece_author_note`. The DECE first-, last- and corresponding-author shares build on these
  flags; for the corresponding author, the affiliations in the correspondence address are tested with
  the same rule.

AUTH affiliation
: A string whose top-level ROR organisation is AUTH or, when it is unmatched, that names Aristotle
  University (or AUTh).

Holder of an affiliation
: An author of the record whose own affiliation text (the Scopus field *Authors with affiliations*)
  contains the affiliation string. Holders are found with the same decomposition of the author's text
  as the DECE-author flag: a string that sits inside a longer string of the same author is held by that
  author too, and an author whose text is empty holds nothing. An affiliation can have several holders,
  some of them DECE authors and some not.

Dual affiliation and DECE-only affiliation
: An author with a *dual affiliation* lists two or more affiliations on the same paper, such as a DECE
  author who also holds a joint, adjunct or visiting position at another institution. Scopus lists all
  of them among the record's affiliations, next to those of the co-authors. A *DECE-only affiliation*
  is an affiliation outside AUTH whose holders (one or more) are all DECE authors: a DECE author's own second
  affiliation, not the affiliation of a co-author based there. As soon as one holder is not a DECE
  author, the affiliation is not DECE-only.

Partner affiliation and partner institution
: A partner affiliation is an affiliation outside AUTH that is matched to ROR, or that has a country
  and names an institution; bare place names and strings that list only departments or laboratories
  (usually units of AUTH itself) do not count. Partner institutions are the distinct top-level ROR
  organisations other than AUTH: {eval}`dp.fmt_int(n_partner_orgs)` in the data,
  {eval}`dp.fmt_int(n_partner_orgs_ext)` of them named by at least one affiliation that is not DECE-only
  (the other {eval}`dp.fmt_int(n_partner_orgs_dece_only)` appear only as DECE authors' own second
  affiliations). A partner institution counts as abroad when an affiliation located outside Greece names it.

Collaboration type
: *International* when at least one affiliation is in a country other than Greece, whether or not it
  is matched to ROR (AUTH strings count as Greece); otherwise *national* when the record has a partner
  affiliation; otherwise *single-institution*, that is AUTH only, possibly several of its units. The
  split is {eval}`dp.fmt_int(collab['single-institution'])` single-institution,
  {eval}`dp.fmt_int(collab['national'])` national and {eval}`dp.fmt_int(collab['international'])`
  international records. For {eval}`dp.fmt_int(n_intl_text_only)` of the international records, the
  only affiliations abroad are unmatched strings whose country was read from the text. These are the
  address-based types (next entry).

Address-based and external-partner indicators
: The collaboration type and the other partner and country indicators (international co-authorship,
  partner institutions and countries, national and industry partners) are *address-based*: every
  affiliation of a record counts, including a DECE author's own second affiliation. This is the
  standard definition (SciVal and the Leiden Ranking also count international collaboration from the
  countries in the affiliation addresses), and it gives the headline figures on every page. The
  *external-partner* variant counts only affiliations of co-authors who are not DECE authors: it leaves
  out the DECE-only affiliations and keeps the AUTH affiliations and every affiliation with at least one
  holder who is not a DECE author. An affiliation without any holder also counts as external, since
  nothing shows that a DECE author holds it. On this basis the split is
  {eval}`dp.fmt_int(collab_ext['single-institution'])` single-institution,
  {eval}`dp.fmt_int(collab_ext['national'])` national and {eval}`dp.fmt_int(collab_ext['international'])`
  international records: {eval}`dp.fmt_int(n_intl_dece_only)` records are international only through
  DECE authors' own second affiliations (DECE-only affiliations; {eval}`dp.fmt_int(n_intl_to_single)` of
  them become single-institution, {eval}`dp.fmt_int(n_intl_to_national)` national), and
  {eval}`dp.fmt_int(n_national_to_single)` national records become single-institution. In {eval}`P_RECENT`, {eval}`dp.fmt_pct(intl_recent)` of the
  records are international on the address basis and {eval}`dp.fmt_pct(intl_recent_ext)` on the
  external-partner basis.

Small groups
: A number of papers that link to an institution, country, region, sector or collaboration type only
  through DECE authors' own second affiliations can rest on one or two DECE authors, and would then in
  practice be their personal paper count. The small-group rule therefore allows such a number to be
  shown, or to follow from the figures shown, only when it is zero or at least {eval}`str(MIN_HOLDERS)`
  distinct DECE authors (Scopus author ids) hold these affiliations. The rule applies to totals over all
  institutions or countries as well as to single years, periods and subgroups, and to numbers that follow
  from other figures, on the same page or across pages, such as the difference between an address-based
  value and its external-partner value, a total minus the parts shown, or a share times its denominator.
  The [Overview](index.ipynb), [Citation impact](impact.ipynb), [Collaboration](collaboration.ipynb),
  [Global reach](geography.ipynb) and [Funding & open access](funding.ipynb) apply it: where a number would
  break the rule, they leave it out, together with enough other values that it cannot be recovered by
  subtraction, or describe it in words. The numbers for all countries together, how many papers change
  collaboration type between the two bases, are chosen once for the whole site, by year, by period and for
  a few spans of years such as all years and {eval}`P_RECENT`, and checked all together, taking into account
  that a value left out stands for at least one paper. On this page every such number, including those that
  follow from the split above, rests on at least {eval}`str(MIN_HOLDERS)` DECE authors, and those for all
  countries together cover all years or {eval}`P_RECENT`.

Sector and industry partner
: The sector of a partner institution is its primary ROR type, taken in the order education,
  healthcare, company, facility, government, nonprofit, archive, other (*funder* only when nothing
  else is listed). A record has an *industry partner* when a matched partner organisation, or its
  top-level organisation, has the ROR type company ({eval}`dp.fmt_int(n_industry)` records;
  {eval}`dp.fmt_int(n_industry_ext)` on the external-partner basis).

Source type
: *Journal* (Scopus document types article, review, letter, editorial, note, erratum, data paper,
  short survey), *conference* (conference paper, conference review), *book* (book, book chapter) or
  *other*. It is derived from the document type, not from the venue, so a conference paper published in
  a journal issue counts as conference.

Open-access category
: From the Scopus *Open Access* field: *gold*, *hybrid* (Scopus "Hybrid Gold"), *bronze* and *green*.
  A record with several routes is counted once, in that order of priority; a record with none is
  *closed*. The field gives the status at the export date, not at publication.

Partial year and periods
: {eval}`str(PARTIAL)` is covered only up to {eval}`as_of_text` ({eval}`dp.fmt_int(n_partial)`
  records, {eval}`dp.fmt_int(n_partial_articles)` of them journal articles: conference proceedings reach
  Scopus months after the event). Charts of yearly counts show it, marked as partial. Shares, means and
  ratios by year leave it out, as do trends, growth rates and period comparisons, so
  {eval}`str(LAST_FULL)` is the latest full year. Totals and all-years shares (as in
  [](#tbl-method-quality)) include it, and the data-quality charts on this page show it as a period of
  its own. Periods are five-year bins that end with the latest full year: *recent* means
  {eval}`P_RECENT`, the latest five full years, the period before it is {eval}`P_PREV`, and so on back
  to {eval}`first_text`. A newer export moves every period along.

Citations and h-index
: Citation counts are the Scopus *Cited by* values at the export date, self-citations included. The
  h-index of a set of papers is the largest number h such that h of the papers have at least h
  citations each.

Normalised citation score
: A paper's citations divided by the mean citations of all records in this data set with the same
  publication year and type, so that 1 is the department's own average for comparable papers.
  Conference papers are compared with conference papers. Articles and reviews are compared together,
  because there are too few reviews for a yearly average of their own ({eval}`dp.fmt_int(n_reviews)` in
  {eval}`str(NCS_FROM)`–{eval}`str(NCS_TO)`, at most {eval}`dp.fmt_int(max_reviews_year)` in a year). It is a
  within-department baseline, not a field-normalised world baseline such as Scopus' Field-Weighted
  Citation Impact, and it averages 1 in every year by construction.

(method-quality)=
## Data quality

How complete are the fields that the pages rely on? [](#tbl-method-quality) lists the share of items
with a value, over all years and for {eval}`P_RECENT`. An empty value is not always a gap: Scopus
leaves the *Open Access* field empty for papers it does not identify as open access, and the funding
field empty when no acknowledgement was indexed.

In [ ]:
#| label: cell-method-quality
dp.table_html(
    quality,
    columns={"field": "Field", "share": "All years", "recent": P_RECENT, "used_on": "Used on"},
    formats={"share": fmt_share, "recent": fmt_share},
    numeric=["share", "recent"],
)

:::{figure} #cell-method-quality
:kind: table
:label: tbl-method-quality
Completeness of key fields: share of items with a value, over all years and in
{eval}`P_RECENT`. Items are records unless the field names another unit; the rows of affiliation
occurrences leave out those that only state a society membership. *Used on* lists the other pages that
depend on the field, directly or through a field derived from it such as the collaboration type.
:::

In [ ]:
#| label: cell-method-completeness
def cell_text(v: float) -> str:
    """Whole percent without the % sign (narrow cells on phones); never 100 (or 0) for a share that is
    not exactly 1 (or 0)."""
    if v != v:
        return ""
    pct = round(v * 100)
    return ">99" if pct == 100 and v < 1 else "<1" if pct == 0 and v > 0 else f"{pct}"


def short_period(p: str) -> str:
    """Period label with its last year cut to two digits (one short line per column); the partial
    year's label stays."""
    m = re.fullmatch(r"(\d{4})–(\d{4})", p)
    return f"{m.group(1)}–{m.group(2)[2:]}" if m else p


labels = [short_period(p) for p in PERIODS]
fig = go.Figure(go.Heatmap(
    z=completeness.to_numpy(), x=labels, y=list(completeness.index),
    zmin=0, zmax=1, colorscale=dp.SEQUENTIAL_SCALE, showscale=False, xgap=2, ygap=2,
    text=completeness.apply(lambda col: col.map(cell_text)).to_numpy(), texttemplate="%{text}",
    textfont=dict(size=12),
    customdata=[[[p, int(n)] for p, n in zip(PERIODS, period_n)] for _ in completeness.index],
    hovertemplate="<b>%{y}</b><br>%{customdata[0]}: %{z:.1%} of %{customdata[1]:,} records<extra></extra>",
))
fig.update_xaxes(showgrid=False, showline=False, side="top", automargin=True)
fig.update_yaxes(showgrid=False, showline=False, autorange="reversed")
fig.update_layout(height=300, margin=dict(t=10, l=10))
fig.show()

:::{figure} #cell-method-completeness
:label: fig-method-completeness
Share of records with a value in key fields (%), by publication period. Years before
{eval}`str(EARLY_END + 1)` are merged (few records); {eval}`str(PARTIAL)` is partial (to {eval}`as_of_short`).
:::

Completeness changes over time ([](#fig-method-completeness)). The publisher is recorded for
{eval}`dp.fmt_pct(publisher_pre)` of the records published before {eval}`str(publisher_from)` and for
{eval}`dp.fmt_pct(publisher_post)` of those from {eval}`str(publisher_from)` to {eval}`str(LAST_FULL)`,
so publisher statistics for older papers depend on fallbacks such as the venue name (see
[Venues & publishers](venues.ipynb)). Funding acknowledgements appear on
{eval}`dp.fmt_pct(funding_earlier)` of the {eval}`P_EARLIER` records and on
{eval}`dp.fmt_pct(funding_recent)` of the {eval}`P_RECENT` records, a rise that may reflect both
more funded research and more complete indexing. A correspondence address with an affiliation, the
basis of the DECE corresponding-author share, is available for {eval}`dp.fmt_pct(corr_all)` of all
records but for only {eval}`dp.fmt_pct(corr_low)` of those from {eval}`P_LOW`, the least complete
period since {eval}`str(MODERN_FROM)`. DOIs are
present for {eval}`dp.fmt_pct(doi_share)` of all records and {eval}`dp.fmt_pct(doi_recent)` of the
recent ones.

**Query precision.** Of the {eval}`dp.fmt_int(n_exported)` records that Scopus returned,
{eval}`dp.fmt_int(n_split)` hold the query words only in different affiliations and were left out
([Data sources](#method-sources)). Of the {eval}`dp.fmt_int(n_pubs)` records kept,
{eval}`dp.fmt_int(n_dece_aff)` ({eval}`dp.fmt_pct(share_dece_aff, 2)`) contain at least one DECE
affiliation string. The other
{eval}`dp.fmt_int(n_no_dece)` (from {eval}`no_dece_years`) {eval}`no_dece_note`, so their department
cannot be checked. The *elec…* rule is broad. A keyword check (strings that name Physics, Journalism or
the medical school and no electrical or computer engineering department) finds
{eval}`dp.fmt_int(n_other_strings)` DECE strings of other AUTH units, such as the Electronics
Laboratory and the Section of Electronics and Computers of the Department of Physics, the Laboratory of
Electronic Media of the Department of Journalism, or units of the medical school. In
{eval}`dp.fmt_int(n_other_unit)` records all DECE strings are of this kind, so these are probably not
DECE papers. The {eval}`dp.fmt_int(n_doubtful)` doubtful records
({eval}`dp.fmt_pct(share_doubtful, 2)` of the total) are kept in all counts. How many DECE papers are
missing cannot be measured from the export itself (see [Limitations](#method-limitations)).

**Affiliation holders.** {eval}`no_holder_text` ([](#tbl-method-quality)).
{eval}`dp.fmt_int(n_dece_only)` occurrences ({eval}`dp.fmt_pct(share_dece_only)` of all,
{eval}`dp.fmt_pct(share_dece_only_ext)` of those outside AUTH) on {eval}`dp.fmt_int(n_dece_only_papers)`
records are DECE-only: {eval}`dp.fmt_int(n_dece_only_abroad)` abroad,
{eval}`dp.fmt_int(n_dece_only_greece)` in Greece and {eval}`dp.fmt_int(n_dece_only_nocountry)` without a
country.

**Document types and stages.** All document types that Scopus indexes are counted, including
{eval}`dp.fmt_int(n_editorial)` editorials and {eval}`dp.fmt_int(n_erratum)` errata, and
{eval}`dp.fmt_int(n_in_press)` records are articles in press whose publication year may still change.

(method-limitations)=
## Limitations

* **Selection by affiliation text.** Records are selected by the query words in their affiliations
  ([Data sources](#method-sources)). DECE papers whose affiliation is written without them (for
  instance only "AUTh", or the former name "University of Thessaloniki"), or split into separate
  department and university affiliations, are missing unless another affiliation of the paper contains
  the query words, and so are papers by staff who gave only another affiliation.
* **Scopus coverage.** Only Scopus-indexed documents are included. Coverage of conference proceedings,
  books and older literature is incomplete, so early annual counts reflect indexing as much as output,
  and the data set is not a complete publication list of the department.
* **Citations** are a snapshot taken on {eval}`as_of_text`, include self-citations and count only
  citing documents indexed in Scopus. Recent papers have had little time to be cited, and research
  fields cite at different rates.
* **Normalised score.** The normalised citation score averages 1 in every year by construction; it
  cannot show trends in impact or impact relative to the world.
* **No per-capita indicators.** No counts are normalised by staff numbers; per-capita productivity would
  need official staff counts per year, and the author profiles mix staff, students and visitors.
* **Author profiles.** Scopus author ids can be split (one person with several ids) or merged. DECE
  status is inferred from affiliation text, and the profiles include doctoral students, post-docs and
  visitors as well as faculty.
* **Affiliation matching.** {eval}`dp.fmt_pct(1 - occ_share_ext)` of the affiliation occurrences
  outside AUTH are unmatched, more in older records; ROR lacks some organisations and parent links; and
  the validation rests on samples of {eval}`dp.fmt_int(r2_n)` strings judged with AI assistance.
* **Affiliation holders.** Holders are read from the per-author affiliation text with the same rule as
  the DECE-author flag and share its limits. An author whose text is empty holds nothing, and a string
  inside a longer string of the same author counts as held by that author; a check on the raw Scopus
  export found that such edge cases hardly change the external-partner figures. DECE status is decided paper by paper, so a
  former DECE member who lists only another institution counts as a co-author who is not a DECE author.
  A DECE author's second affiliation can also reflect a real link with that institution, which is why
  the address-based figures remain the headline and the external-partner figures complement them.
* **Open access and funding.** Open-access flags give Scopus' current, Unpaywall-based status, not the
  status at publication. Funding data cover only the acknowledgements that Scopus indexed.
* **Topics** are inferred from titles alone.
* **Associations, not causes.** Comparisons between groups of papers (for example funded and unfunded
  papers) describe associations; they do not control for research field, team size or other
  differences.
* **Unofficial.** This is an independent analysis, not an official publication of AUTH. By design the
  site shows no author names, e-mail addresses, author ids or per-person metrics.